# 군집화 과제 — 와인 데이터

**제출** GitHub `develop` 브랜치에 push · **데이터** `sklearn.datasets.load_wine`

---

수업 실습에서는 **정답이 없는** 고객 데이터를 다뤘습니다.
이번 과제는 **정답이 있는** 데이터입니다. 와인 178병이 실제로 **3종류**로 나뉘어 있어요.

정답이 있으면 좋은 게 하나 있습니다. **군집화가 실제로 맞았는지 채점할 수 있다**는 것.
그래서 이번 과제의 진짜 주제는 이겁니다.

> ### 지표가 좋다고 군집화가 잘 된 걸까?

---

| | 내용 |
|---|---|
| **Part 1** | 코드를 읽고 **주석 달기** |
| **Part 2** | 결과를 보고 **해석하기** |
| **Part 3** | 직접 판단하고 **근거 쓰기** |

> 답을 적는 칸은 **마크다운 셀**입니다. 길게 쓸 필요 없습니다. **근거를 숫자로 대면** 충분합니다.

## 0. 준비

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, confusion_matrix, adjusted_rand_score
from scipy.optimize import linear_sum_assignment

import warnings; warnings.filterwarnings('ignore')

wine = load_wine()
X_raw = pd.DataFrame(wine.data, columns=wine.feature_names)
y_true = wine.target                       # 실제 와인 종류 (0, 1, 2)

print(f'와인 {X_raw.shape[0]}병 · 화학 성분 {X_raw.shape[1]}개 · 실제 종류 {len(set(y_true))}가지')
X_raw.head()

와인 178병 · 화학 성분 13개 · 실제 종류 3가지


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


군집화 결과를 **정답과 맞춰보는** 함수입니다. 그대로 실행하시면 됩니다.

군집 번호(0,1,2)와 정답 번호(0,1,2)는 **순서가 다를 수 있어서**, 가장 잘 맞는 짝을 찾아 채점합니다.

In [3]:
def match_accuracy(y_true, labels):
    """군집 번호와 정답 번호를 최적으로 짝지어 일치율을 계산"""
    mask = labels != -1                       # 노이즈(-1)는 오답 처리
    if mask.sum() == 0:
        return 0.0
    cm = confusion_matrix(y_true[mask], labels[mask])
    row, col = linear_sum_assignment(-cm)
    return cm[row, col].sum() / len(y_true)

---
# Part 1. 코드를 읽고 주석 달기

아래는 **군집화 파이프라인 전체**입니다. 지금은 주석이 없습니다.

In [4]:
X_scaled = StandardScaler().fit_transform(X_raw)                          # ①

pca = PCA(n_components=2)                                                # ②
X_pca = pca.fit_transform(X_scaled)                                       # ③

kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)  # ④
labels = kmeans.fit_predict(X_scaled)                                     # ⑤

sil = silhouette_score(X_scaled, labels)                                  # ⑥
acc = match_accuracy(y_true, labels)                                      # ⑦

print(f'실루엣 {sil:.3f} · 정답 일치 {acc:.3f}')

  File "c:\Users\thdus\.anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\thdus\.anaconda\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\thdus\.anaconda\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\thdus\.anaconda\Lib\subprocess.

실루엣 0.285 · 정답 일치 0.966


### 1-1. 각 줄이 하는 일을 채우세요

`무엇을 하는가`뿐 아니라 **`왜 필요한가`** 를 꼭 적어주세요. 이게 핵심입니다.

| 줄 | 무엇을 하는가 | 왜 필요한가 |
|---|---|---|
| ① | 데이터 표준화 |변수들의 단위를 맞춰서 단위 큰 변수가 큰 영향을 주지 않게 하기 위해|
| ② | 피처들을 2개의 주성분으로 줄임 | 차원 축소를 하기 위해 |
| ③ |학습과 적용을 한번에 함|1에서 표준화한 데이터를 학습해서 2의 결과로 만들 수 있게 함|
| ④ | 군집화를 k-means로 진행함, 3개의 군집으로 나누고, 적절히 떨어진 위치에 초기 중심점을 잡고, k-means를 10번 실행함, 난수는 42로 고정| 데이터를 3개로 군집화하기 위한 모델을 설정 |
| ⑤ | 실제 군집화를 진행함, 각 데이터의 중심점을 찾고, 데이터가 어떤 군집에 속할 지를 결정함| 4에서 진행한 군집화 모델을 사용하기 위해|
| ⑥ | 5에서 만든 군집의 성능 평가 | 군집화 모델의 성능을 평가하기 위해 |
| ⑦ | 군집 번호와 정답 번호 비교해서 일치율 확인 | 정확도 평가를 위해 |

### 1-2. 순서에 대한 질문

**(a)** 표준화하지 않은 `X_raw`에 PCA를 먼저 적용하면 어떻게 될까요? 왜 그런가요?
<br>
A. 표준화를 하지 않으면 단위가 큰 변수가 PCA에 큰 영향을 줘서 잘못된 방향으로 진행될 수 있음
<br>

**(b)** ⑤를 보면 군집화에 `X_pca` 가 아니라 **`X_scaled` 를 넣었습니다.** ③에서 PCA를 해놓고 왜 안 썼을까요?
<br>
A. 차원축소(X_pca)과정에서 정보가 손실되기 때문에 원본 데이터를 담은 X_scaled를 사용해야 합니다. PCA를 사용하지 않은 이유는 잘 모르겠으나 차원 축소를 하면 그래프로 나타낼 때 수월하기 때문에 시각화용으로 pca를 구한 게 아닐까 싶습니다. 
<br>

**(c)** ④의 `random_state=42` 를 지우면 무슨 일이 생기나요? k-means의 어떤 성질 때문인가요?

<br>
A. 초기 중심점을 잡을 때 난수가 고정이 안되어서 군집화를 진행할 때마다 초기 중심점이 달라져서 결과가 달라집니다. 

---
# Part 2. 결과를 보고 해석하기

## 2-1. 차원을 줄였더니 점수가 올랐습니다

In [5]:
rows = []
labels_by_space = {}
for name, data in [('13차원 (원본)', X_scaled), ('2차원 (PCA)', X_pca)]:
    lab = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(data)
    labels_by_space[name] = lab
    rows.append({'데이터': name,
                 '실루엣': round(silhouette_score(data, lab), 3),
                 '정답 일치율': round(match_accuracy(y_true, lab), 3)})

labels_13d = labels_by_space['13차원 (원본)']
labels_2d = labels_by_space['2차원 (PCA)']
cm_between = confusion_matrix(labels_13d, labels_2d)
row, col = linear_sum_assignment(-cm_between)
n_different = len(labels_13d) - cm_between[row, col].sum()
ari_between = adjusted_rand_score(labels_13d, labels_2d)

print(f'두 군집 결과 간 ARI: {ari_between:.3f}')
print(f'군집 번호를 최적으로 맞춘 뒤 배정이 다른 와인: {n_different}병')
pd.DataFrame(rows)

두 군집 결과 간 ARI: 0.932
군집 번호를 최적으로 맞춘 뒤 배정이 다른 와인: 4병


,데이터,실루엣,정답 일치율
0,13차원 (원본),0.285,0.966
1,2차원 (PCA),0.561,0.966


### 2-1

실루엣은 **0.285 → 0.561로 거의 2배**가 되었고, 정답 일치율은 두 경우 모두 96.6%입니다.

ARI(조정 랜드 지수)는 군집 번호와 무관하게 두 군집 결과가 얼마나 같은지 재며, **1이면 완전히 같은 분할**을 뜻합니다.

**(a)** 정답 일치율이 같다는 사실만으로 두 군집 결과가 완전히 같다고 할 수 있을까요? 위에 출력된 ARI와 배정이 다른 와인 수를 근거로 설명하세요.

<br>
A. 완전히 같다고 할 순 없습니다.
일단 ARI가 0.932로 1이 아니며, 군집 번호를 최적으로 맞춘 후에도 4병이 기존과 다른 군집에 배정되었기 때문에 정답 일치율이 같다고 해서 두 군집의 결과가 같다고 할 수 없다. 
<br>

**(b)** 정답 일치율은 같은데 실루엣은 크게 오른 이유는 무엇일까요?
*(힌트: 실루엣은 무엇을 재나요? 그리고 강의 `2-4`의 **차원의 저주**에서 거리는 어떻게 되었나요?)*

<br>
A. 정답 일치율은 정답 클래스와 얼마나 일치하는 지를 보는 것이고, 실루엣은 데이터간 거리로 같은 군집끼리의 응집도와 다른 군집간의 분리도를 보는 것입니다. 따라서 차원이 높은 원본에서는 실루엣 계수값이 낮고, 차원이 적은 PCA에선 실루엣 계수값이 높습니다. 이는 각 군집 간 응집도와 분리도가 차원축소를 통해 개선되어 군집이 더 뚜렷해져서 나타나는 현상입니다.
<br>

**(c)** 그렇다면 **"실루엣 0.5 이상이면 좋은 군집화"** 같은 절대 기준을 쓸 수 있을까요?

<br>
A. b의 경우에서 정답 일치율은 같은데 차원축소를 했다고 실루엣 계수 값이 커졌습니다. 실루엣이 거리 기반으로 계산되기 때문에 스케일링 방식이나 PCA 여부에 따라서 값이 달라집니다. 따라서 절대적인 기준으로는 무리가 있어보입니다.

## 2-2. 실루엣 0.285는 나쁜 걸까요?

수업 실습(고객 데이터)에서는 실루엣이 **0.481**이었습니다.
지금 와인 13차원은 **0.285** 로 훨씬 낮습니다. 그런데 정답은 **96.6%** 맞혔어요.

### 2-2

**(a)** 실루엣 0.285를 보고 *"군집화에 실패했다"* 고 판단하면 무엇을 놓치는 건가요?

<br>
A. 정답 일치율은 96.6%이지만 고차원이기에 거리기반의 지표에서 좋은 점수를 받지 못한 것인데 이 거리기반 지표만을 갖고 군집화에 실패했다고 판단한다면, 정답과의 일치여부를 놓치게 되는 것입니다.
<br>
**(b)** 정답 레이블이 **없는** 상황(수업의 고객 데이터)에서는 이 함정을 어떻게 피할 수 있을까요?
*(힌트: 실습 4에서 k=3과 k=4 중 무엇을 보고 골랐나요?)*

<br>
A. 실루엣 계수값만으로 비교하기 보다는 각 군집의 특성이나 군집 간 차이와 같은 점들을 고려해서 더 군집의 특성을 잘 나타낼 수 있는 k를 구하면 됩니다.

## 2-3. DBSCAN이 최고 점수를 받았습니다

`eps` 를 바꿔가며 DBSCAN을 돌리고, **노이즈를 제외하고** 실루엣을 계산했습니다.

In [6]:
rows = []
for eps in [0.3, 0.4, 0.5, 0.6, 0.8]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(X_pca)
    n_cluster = len(set(lab)) - (1 if -1 in lab else 0)
    mask = lab != -1
    sil = silhouette_score(X_pca[mask], lab[mask]) if n_cluster > 1 else np.nan
    rows.append({'eps': eps, '군집 수': n_cluster,
                 '노이즈': (~mask).sum(),
                 '노이즈 비율(%)': round((~mask).mean()*100, 1),
                 '실루엣(노이즈 제외)': round(sil, 3),
                 '정답 일치율': round(match_accuracy(y_true, lab), 3)})
pd.DataFrame(rows)

,eps,군집 수,노이즈,노이즈 비율(%),실루엣(노이즈 제외),정답 일치율
0,0.3,5,115,64.6,0.790,0.275
1,0.4,9,60,33.7,0.498,0.404
2,0.5,5,31,17.4,0.523,0.573
3,0.6,2,16,9.0,0.499,0.607
4,0.8,2,5,2.8,0.478,0.635


### 2-3

`eps=0.3` 일 때 실루엣이 **0.790** 으로 표에서 가장 높습니다. Part 2-1의 k-means(0.561)보다도 높아요.

**(a)** 그런데 이 설정은 **178병 중 115병(64.6%)을 노이즈로 버렸습니다.**
노이즈를 제외하고 실루엣을 계산하면 왜 점수가 높게 나올까요?
<br>
A. 군집에 넣기 애매한 것들이 노이즈가 되는데 경계에 있는 애매한 것들이 버려지기 때문에 군집이 더 뚜렷해집니다. 따라서 점수가 높게 나옵니다.
<br>

**(b)** 정답 일치율 컬럼을 함께 보세요. 실루엣 1등과 정답 일치율 1등이 같은가요?
<br>
A. 실루엣 1등은 eps=0.3 일 때고, 정답 일치율 1등은 eps=0.8 일 때 입니다. 따라서 실루엣 1등과 정답 일치율 1등이 같지 않습니다. 
<br>

**(c)** 이 표를 근거로, **DBSCAN의 실루엣을 k-means의 실루엣과 나란히 비교하면 안 되는 이유**를 한 문장으로 쓰세요.

<br> 
A. DBSCAN은 노이즈를 제외한 데이터의 실루엣 계수값을 구한 것이고, k-means는 모든 데이터를 대상으로 실루엣 계수를 구한 것입니다. 데이터의 범위 자체가 다르기 때문에 나란히 비교하기엔 적절하지 않습니다.

---
# Part 3. 직접 판단하고 근거 쓰기

이제 **여러분이 정하세요.** 정답을 맞히는 게 아니라 **근거를 대는 것**이 이 파트의 전부입니다.

## 3-1. k를 정하세요

In [12]:
rows = []
for k in range(2, 7):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    rows.append({'k': k,
                 '실루엣': round(silhouette_score(X_scaled, lab), 3),
                 '정답 일치율': round(match_accuracy(y_true, lab), 3)})
pd.DataFrame(rows)

,k,실루엣,정답 일치율
0,2,0.259,0.601
1,3,0.285,0.966
2,4,0.260,0.831
3,5,0.202,0.657
4,6,0.237,0.713


### 3-1

**(a)** 실루엣만 보고 k를 고른다면 몇인가요? 정답 일치율만 보면 몇인가요?
<br>
A. 실루엣만 본다면 k=3 이고, 정답 일치율만 본다면 k=3입니다.
<br>

**(b)** 만약 **정답 레이블이 없었다면** — 즉 오른쪽 컬럼을 볼 수 없었다면 — 여러분은 k를 몇으로 정했을까요?
그 판단이 결과적으로 맞았나요?
<br>
A. 실루엣 계수도 고려하고 군집의 특징도 고려하여 정했을 것입니다. 현재로썬 실루엣 계수만 고려해야하므로 k=3을 택했을 것 같습니다. 결과적으로 맞는 선택이긴 합니다.

## 3-2. 알고리즘을 고르세요

아래 셀의 `my_model` 을 직접 바꿔가며 최소 **3가지**를 시도해보세요.

In [16]:
# 여기를 바꿔가며 실행하세요
# my_model = KMeans(n_clusters=3, n_init=10, random_state=42)

# 시도해볼 것들:
# my_model = GaussianMixture(n_components=3, random_state=42)
# my_model =AgglomerativeClustering(n_clusters=3, linkage='ward')
# my_model =AgglomerativeClustering(n_clusters=3, linkage='single')
my_model =DBSCAN(eps=2.2, min_samples=5)
#     ※ 2-3에서는 2차원(X_pca)에 eps=0.6을 썼지만 여기는 13차원(X_scaled)입니다.
#        차원이 높아지면 점들 사이 거리가 전반적으로 멀어져서 eps도 훨씬 크게 잡아야 해요.
#        (eps=0.6을 그대로 넣으면 178병이 전부 노이즈가 됩니다. 직접 확인해 보세요.)

labels = my_model.fit_predict(X_scaled)
mask = labels != -1
n_cluster = len(set(labels)) - (1 if -1 in labels else 0)

print(f'모델          : {type(my_model).__name__}')
print(f'군집 수       : {n_cluster}')
print(f'노이즈        : {(~mask).sum()}개')
if n_cluster > 1:
    print(f'실루엣        : {silhouette_score(X_scaled[mask], labels[mask]):.3f}')
else:
    print(f'실루엣        : 계산 불가 — 군집이 {n_cluster}개뿐입니다 (eps를 키워보세요)')
print(f'정답 일치율   : {match_accuracy(y_true, labels):.3f}')

모델          : DBSCAN
군집 수       : 2
노이즈        : 55개
실루엣        : 0.348
정답 일치율   : 0.483


### 3-2

시도한 결과를 채우세요.

| 알고리즘 | 군집 수 | 실루엣 | 정답 일치율 |
|---|---|---|---|
| KMeans| 3|0.285 | 0.966|
|GaussianMixture | 3| 0.285| 0.966|
| AgglomerativeClustering(ward)| 3|  0.277| 0.927|
| AgglomerativeClustering(single)|3 |0.183 |0.376 |
| DBSCAN| 2| 0.348| 0.483|


**(a)** 가장 좋다고 판단한 알고리즘은 무엇이고, **무엇을 근거로** 골랐나요?

<br>
정답 일치율과 실루엣 1등은 Kmeans와 GaussianMixture입니다. 따라서 Kmeans와 GaussianMixture 를 가장 좋은 알고리즘이라고 판단했습니다.

**(b)** `linkage='single'` 을 넣어보셨나요? 결과가 왜 그렇게 나왔는지 **실습 2에서 본 것**과 연결해 설명하세요.

<br> 네.. 
single, 즉 단일 연결법의 경우에 가장 가까운 한 쌍만 보고 병합합니다. 따라서 점들이 가까이 이어져있으면 체인효과에 따라 하나로 묶여지고, 너무 멀리 있으면 안붙어서 1개짜리 군집이 됩니다. 따라서 데이터 내에 체인효과와 같은 현상이 일어나서 실루엣 계수값이 줄어들고, 정답 일치율이 줄어든 것 같습니다.


## 3-3. 마지막 — 군집에 이름 붙이기

In [17]:
# 3-2에서 가장 좋다고 판단한 모델로 바꾸세요.
# 아래에는 KMeans를 기본값으로 두었습니다.
best_model = GaussianMixture(n_components=3, random_state=42)
best_labels = best_model.fit_predict(X_scaled)

t = X_raw.copy(); t['군집'] = best_labels
t.groupby('군집')[['alcohol', 'flavanoids', 'color_intensity', 'proline', 'hue']].mean().round(2)

,alcohol,flavanoids,color_intensity,proline,hue
군집,,,,,
0,12.25,2.05,2.97,510.17,1.06
1,13.13,0.82,7.23,619.06,0.69
2,13.68,3.00,5.45,1100.23,1.07


### 3-3

**(a)** 3-2에서 선택한 모델의 결과로 나온 세 군집의 특징을 한 줄씩 쓰고, **이름을 붙여보세요.**
alcohol : 알코올
flavanoids : 플라보노이드 함량 
color_intensity : 색 얼마나 진한지
proline : 프롤린 함량
hue : 색상의 특성 : 작을수록 자주색 높을 수록 노란빛

| 군집 | 특징 | 이름 |
|---|---|---|
| 0 | 군집들 중에서 alcohol과 color_intensity, proline의 값이 가장 낮음, hue가 높은 편| 저알코올, 연한 노란색의 저프롤린 와인|
| 1 | flavanoids가 가장 작고, color_intensity가 가장 높지만 hue는 젤 작음| 저플라보노이드, 진한 자줏빛 와인 |
| 2 | alcohol이 가장 높고, flavanoids도 가장 높고, proline과 hue도 가장 높음 | 도수가 높고, 고플라보노이드, 고프롤라인의 노란빛 와인 |

<br>

**(b)** 수업 실습에서는 *"군집에 이름을 붙일 수 없다면 잘못 나눈 것"* 이라고 했습니다.
와인 데이터에서 이름을 붙이는 것과, 고객 데이터에서 이름을 붙이는 것은 **무엇이 다른가요?**

<br> 와인 데이터에서는 화학적인 특징을 담은 피처가 있는데 이 피처의 값이 높다고 해서 어떤 맛인지를 판단하거나, 품질의 정도가 어떤 지를 알 수 없습니다. 하지만 고객 데이터에서는 피처의 특징이 곧바로 실무에 맞닿게 되는 (ex: 연소득이 높다면 고소득..) 차이점이 있습니다.

---
<div style="padding:16px;border-radius:8px">

### 이 과제가 물어본 것

처음의 질문으로 돌아갑니다.

> **지표가 좋다고 군집화가 잘 된 걸까?**

- **2-1** — PCA 전후의 정답 일치율이 같아도 군집 배정은 완전히 같지 않으며, 실루엣은 표현 공간에 따라 크게 바뀌 수 있습니다.
- **2-2** — 실루엣 0.285가 정답 96.6%를 맞혔습니다.
- **2-3** — 데이터의 3분의 2를 버리면 실루엣이 최고가 됩니다.

**지표는 후보를 좁혀줄 뿐입니다. 최종 판단은 여러분이 합니다.**

</div>

수고하셨습니다!